In [ ]:
# ==========================
# Enhanced Flask XGBoost App with Professional UI, Advanced Visualization & Applicability Domain
# ==========================

from flask import Flask, render_template, request, jsonify, send_from_directory
import pandas as pd
import joblib
import os
import webbrowser
from threading import Timer
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import GridSearchCV, train_test_split
from xgboost import XGBRegressor
import numpy as np
import logging
from datetime import datetime
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import io
import base64
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

app = Flask(__name__)

# === Configuration ===
class Config:
    MODEL_PATH = 'xgboost_grout_model.pkl'
    FEATURES_PATH = 'model_features.pkl'
    SCALER_PATH = 'scaler.pkl'
    STATS_PATH = 'feature_stats.pkl'
    LABEL_ENCODERS_PATH = 'label_encoders.pkl'
    TEMPLATE_DIR = 'templates'
    HTML_FILE = os.path.join(TEMPLATE_DIR, 'index.html')
    DATA_PATH = r'D:\2025 Work\Dr Ebrahim Sharifi\Grout Data\Data\finnal grout data set-20250421.csv'
    LOG_FILE = 'app_log.txt'
    SAVED_INPUTS_FILE = 'saved_inputs.json'
    PLOTS_DIR = 'static/plots'

# === Setup logging ===
def setup_logging():
    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s - %(levelname)s - %(message)s',
        handlers=[
            logging.FileHandler(Config.LOG_FILE, encoding='utf-8'),
            logging.StreamHandler()
        ]
    )

setup_logging()

# === Create directories ===
def create_directories():
    """Create necessary directories for the application."""
    os.makedirs(Config.TEMPLATE_DIR, exist_ok=True)
    os.makedirs(Config.PLOTS_DIR, exist_ok=True)
    os.makedirs('static/css', exist_ok=True)
    os.makedirs('static/js', exist_ok=True)

create_directories()

# === Global variables for model components ===
model = None
feature_names = []
scaler = None
feature_stats = {}
label_encoders = {}
best_params = {}
model_date = "Unknown"

# === Function to validate inputs against applicability domain ===
def validate_input_domain(input_values, feature_stats, label_encoders):
    """
    Validate that inputs fall within the model's applicability domain.
    Returns warnings and out-of-range information.
    """
    warnings = []
    out_of_range_inputs = {}
    
    for feature, value in input_values.items():
        stats = feature_stats.get(feature, {})
        
        if not stats.get('is_categorical', False):
            min_val = stats.get('min')
            max_val = stats.get('max')
            
            if min_val is not None and max_val is not None:
                if value < min_val or value > max_val:
                    out_of_range_inputs[feature] = {
                        'value': value,
                        'min': min_val,
                        'max': max_val
                    }
                    warnings.append(
                        f"{feature}: {value:.2f} is outside training range "
                        f"[{min_val:.2f}, {max_val:.2f}]"
                    )
        else:
            # For categorical features, check if value is in training classes
            if feature in label_encoders:
                valid_categories = label_encoders[feature].classes_.tolist()
                # Convert value to string for comparison if needed
                str_value = str(value)
                if str_value not in valid_categories and str(value) not in valid_categories:
                    # Show first 5 categories as example
                    example_cats = ', '.join(map(str, valid_categories[:5]))
                    if len(valid_categories) > 5:
                        example_cats += f" ... ({len(valid_categories)} total)"
                    warnings.append(
                        f"{feature}: '{value}' is not in training categories "
                        f"(valid examples: {example_cats})"
                    )
    
    return warnings, out_of_range_inputs

# === Enhanced Model Training with GridSearchCV ===
def create_model_from_csv():
    """Train and save enhanced XGBoost model using GridSearchCV."""
    global model, feature_names, scaler, feature_stats, label_encoders, best_params, model_date
    
    logging.info("Training enhanced model with GridSearchCV...")
    
    if not os.path.exists(Config.DATA_PATH):
        raise FileNotFoundError("Data file not found at: " + Config.DATA_PATH)
    
    df = pd.read_csv(Config.DATA_PATH)
    logging.info("Loaded dataset with shape: " + str(df.shape))
    
    if 'TAKE' not in df.columns:
        raise ValueError("'TAKE' column missing in dataset!")

    X = df.drop(columns=['TAKE'])
    y = df['TAKE']
    feature_names = X.columns.tolist()
    
    # Define which features are numerical (continuous)
    # Based on the CSV data, these should ALL be numerical
    numerical_features = ['depth', 'Q', 'LU', 'app', 'Pg', 'w/c', 'S_5N']
    
    # Enhanced feature statistics with actual data values
    feature_stats = {}
    label_encoders = {}
    
    # Preprocess features with enhanced handling
    X_processed = X.copy()
    for feature in feature_names:
        col_data = X[feature].dropna()
        
        # Force all features to be numerical since they're all numeric in the CSV
        is_categorical = False
        
        if not is_categorical:
            # For numerical features, store proper statistics
            feature_stats[feature] = {
                'is_categorical': False,
                'data_type': 'numerical',
                'min': float(col_data.min()),
                'max': float(col_data.max()),
                'mean': float(col_data.mean()),
                'std': float(col_data.std()),
                'q1': float(col_data.quantile(0.25)),
                'q3': float(col_data.quantile(0.75)),
                'median': float(col_data.median()),
                'unique_count': col_data.nunique(),
                'count': len(col_data)
            }
            logging.info(f"Feature '{feature}' treated as numerical with range [{col_data.min():.2f}, {col_data.max():.2f}]")
    
    # Enhanced missing value handling (fill with median for numerical)
    for feature in X_processed.columns:
        if X_processed[feature].isnull().any():
            X_processed[feature].fillna(X_processed[feature].median(), inplace=True)

    # Enhanced scaling
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_processed)

    # Split data for validation
    X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

    # Enhanced GridSearchCV for hyperparameter tuning
    param_grid = {
        'n_estimators': [100, 200, 300],
        'max_depth': [3, 6, 9],
        'learning_rate': [0.01, 0.1, 0.2],
        'subsample': [0.8, 0.9, 1.0],
        'colsample_bytree': [0.8, 0.9, 1.0]
    }

    model = XGBRegressor(random_state=42)
    grid_search = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        cv=5,
        scoring='neg_mean_squared_error',
        n_jobs=-1,
        verbose=1
    )

    logging.info("Starting GridSearchCV...")
    grid_search.fit(X_train, y_train)
    
    # Get best model
    best_model = grid_search.best_estimator_
    
    # Generate model evaluation plots
    generate_model_plots(best_model, X_test, y_test, feature_names, grid_search)

    logging.info(f"GridSearchCV completed. Best parameters: {grid_search.best_params_}")
    logging.info(f"Best CV score: {-grid_search.best_score_:.4f}")

    # Save all components
    joblib.dump(best_model, Config.MODEL_PATH)
    joblib.dump(feature_names, Config.FEATURES_PATH)
    joblib.dump(scaler, Config.SCALER_PATH)
    joblib.dump(feature_stats, Config.STATS_PATH)
    joblib.dump(label_encoders, Config.LABEL_ENCODERS_PATH)
    
    best_params = grid_search.best_params_
    model_date = datetime.now().strftime('%Y-%m-%d %H:%M')
    
    logging.info(f"Enhanced model trained and saved with {len(feature_names)} features.")
    return best_model, feature_names, scaler, feature_stats, label_encoders, best_params

def generate_model_plots(model, X_test, y_test, feature_names, grid_search):
    """Generate comprehensive model evaluation plots."""
    try:
        # Create plots directory
        os.makedirs(Config.PLOTS_DIR, exist_ok=True)
        
        # Predictions vs Actual
        y_pred = model.predict(X_test)
        
        plt.style.use('default')
        fig, axes = plt.subplots(2, 2, figsize=(15, 12))
        
        # Plot 1: Predictions vs Actual
        axes[0, 0].scatter(y_test, y_pred, alpha=0.6, color='blue')
        axes[0, 0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
        axes[0, 0].set_xlabel('Actual Values')
        axes[0, 0].set_ylabel('Predicted Values')
        axes[0, 0].set_title('Predictions vs Actual Values')
        r2 = r2_score(y_test, y_pred)
        axes[0, 0].text(0.05, 0.95, f'R² = {r2:.3f}', transform=axes[0, 0].transAxes, 
                       bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))
        
        # Plot 2: Residuals
        residuals = y_test - y_pred
        axes[0, 1].scatter(y_pred, residuals, alpha=0.6, color='green')
        axes[0, 1].axhline(y=0, color='r', linestyle='--')
        axes[0, 1].set_xlabel('Predicted Values')
        axes[0, 1].set_ylabel('Residuals')
        axes[0, 1].set_title('Residual Plot')
        
        # Plot 3: Feature Importance
        if hasattr(model, 'feature_importances_'):
            importance = model.feature_importances_
            indices = np.argsort(importance)[::-1]
            top_features = min(10, len(feature_names))
            
            axes[1, 0].barh(range(top_features), importance[indices[:top_features]])
            axes[1, 0].set_yticks(range(top_features))
            axes[1, 0].set_yticklabels([feature_names[i] for i in indices[:top_features]])
            axes[1, 0].set_xlabel('Feature Importance')
            axes[1, 0].set_title('Top 10 Feature Importances')
        
        # Plot 4: Error Distribution
        axes[1, 1].hist(residuals, bins=30, alpha=0.7, color='orange', edgecolor='black')
        axes[1, 1].axvline(x=0, color='r', linestyle='--')
        axes[1, 1].set_xlabel('Residuals')
        axes[1, 1].set_ylabel('Frequency')
        axes[1, 1].set_title('Error Distribution')
        
        plt.tight_layout()
        plt.savefig(os.path.join(Config.PLOTS_DIR, 'model_evaluation.png'), dpi=300, bbox_inches='tight')
        plt.close()
        
        # Parameter performance plot
        if hasattr(grid_search, 'cv_results_'):
            plot_parameter_performance(grid_search)
            
    except Exception as e:
        logging.error(f"Error generating plots: {str(e)}")

def plot_parameter_performance(grid_search):
    """Plot parameter performance from GridSearchCV."""
    try:
        results = grid_search.cv_results_
        param_names = list(grid_search.best_params_.keys())
        
        fig, axes = plt.subplots(2, 2, figsize=(12, 10))
        axes = axes.ravel()
        
        for i, param in enumerate(param_names[:4]):
            if param in results:
                # Get unique parameter values and their mean scores
                param_values = results[f'param_{param}'].data
                unique_values = sorted(set(param_values))
                mean_scores = []
                
                for value in unique_values:
                    mask = param_values == value
                    if mask.any():
                        mean_scores.append(-results['mean_test_score'][mask].mean())
                
                axes[i].plot(unique_values, mean_scores, 'bo-', markersize=8)
                axes[i].set_xlabel(param)
                axes[i].set_ylabel('Negative MSE')
                axes[i].set_title(f'Parameter: {param}')
                axes[i].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(os.path.join(Config.PLOTS_DIR, 'parameter_performance.png'), dpi=300, bbox_inches='tight')
        plt.close()
        
    except Exception as e:
        logging.error(f"Error plotting parameter performance: {str(e)}")

# === Professional HTML Template with Enhanced UI ===
def create_html_template(feature_names, feature_stats, best_params=None):
    """Create a professional grid-styled HTML input page with advanced visualization."""
    
    html = """<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Advanced Grout TAKE Prediction System</title>
    <link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.0.0/css/all.min.css">
    <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0/dist/css/bootstrap.min.css" rel="stylesheet">
    <script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
    <style>
        :root {
            --primary: #2c5aa0;
            --secondary: #1e3a5f;
            --success: #28a745;
            --warning: #ffc107;
            --danger: #dc3545;
            --info: #17a2b8;
            --dark: #343a40;
            --light: #f8f9fa;
            --gradient-primary: linear-gradient(135deg, #2c5aa0 0%, #1e3a5f 100%);
            --gradient-success: linear-gradient(135deg, #28a745 0%, #20c997 100%);
            --gradient-warning: linear-gradient(135deg, #ffc107 0%, #fd7e14 100%);
        }
        
        body {
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            min-height: 100vh;
            padding: 20px;
            background-attachment: fixed;
        }
        
        .glass-card {
            background: rgba(255, 255, 255, 0.95);
            backdrop-filter: blur(20px);
            border-radius: 20px;
            border: 1px solid rgba(255, 255, 255, 0.3);
            box-shadow: 0 15px 35px rgba(0,0,0,0.1);
            overflow: hidden;
        }
        
        .dashboard-header {
            background: var(--gradient-primary);
            color: white;
            padding: 40px 30px;
            border-radius: 20px 20px 0 0;
            margin: -30px -30px 30px -30px;
            position: relative;
            overflow: hidden;
        }
        
        .dashboard-header::before {
            content: '';
            position: absolute;
            top: -50%;
            right: -50%;
            width: 100%;
            height: 100%;
            background: radial-gradient(circle, rgba(255,255,255,0.1) 1px, transparent 1px);
            background-size: 20px 20px;
            transform: rotate(30deg);
        }
        
        .metric-card {
            background: white;
            border-radius: 15px;
            padding: 25px;
            margin-bottom: 25px;
            box-shadow: 0 8px 25px rgba(0,0,0,0.1);
            border-left: 5px solid var(--primary);
            transition: all 0.3s cubic-bezier(0.4, 0, 0.2, 1);
            position: relative;
            overflow: hidden;
        }
        
        .metric-card::before {
            content: '';
            position: absolute;
            top: 0;
            left: 0;
            right: 0;
            height: 4px;
            background: var(--gradient-primary);
        }
        
        .metric-card:hover {
            transform: translateY(-8px);
            box-shadow: 0 15px 35px rgba(0,0,0,0.15);
        }
        
        .metric-value {
            font-size: 2.5rem;
            font-weight: 800;
            color: var(--secondary);
            margin-bottom: 5px;
        }
        
        .metric-label {
            font-size: 0.95rem;
            color: #6c757d;
            text-transform: uppercase;
            letter-spacing: 1.5px;
            font-weight: 600;
        }
        
        .input-section {
            background: white;
            border-radius: 15px;
            padding: 25px;
            margin-bottom: 25px;
            border: 2px solid #e9ecef;
            transition: all 0.3s ease;
            position: relative;
        }
        
        .input-section:hover {
            border-color: var(--primary);
            box-shadow: 0 8px 25px rgba(44, 90, 160, 0.15);
        }
        
        .input-section.required {
            border-left: 5px solid var(--danger);
            background: linear-gradient(135deg, #fff5f5 0%, #fff 100%);
        }
        
        .prediction-result {
            background: var(--gradient-success);
            color: white;
            padding: 50px 30px;
            border-radius: 20px;
            text-align: center;
            margin: 30px 0;
            animation: fadeInUp 0.8s ease;
            box-shadow: 0 10px 30px rgba(40, 167, 69, 0.3);
            position: relative;
            overflow: hidden;
        }
        
        .prediction-result::before {
            content: '';
            position: absolute;
            top: -20px;
            right: -20px;
            width: 100px;
            height: 100px;
            background: rgba(255,255,255,0.1);
            border-radius: 50%;
        }
        
        @keyframes fadeInUp {
            from {
                opacity: 0;
                transform: translateY(40px);
            }
            to {
                opacity: 1;
                transform: translateY(0);
            }
        }
        
        .feature-importance-chart {
            background: white;
            border-radius: 15px;
            padding: 25px;
            margin: 25px 0;
            box-shadow: 0 8px 25px rgba(0,0,0,0.1);
        }
        
        .model-stats {
            background: var(--gradient-primary);
            color: white;
            padding: 30px;
            border-radius: 15px;
            margin: 25px 0;
            box-shadow: 0 8px 25px rgba(0,0,0,0.15);
        }
        
        .tab-content {
            background: white;
            border-radius: 0 0 20px 20px;
            padding: 30px;
            border: 1px solid #dee2e6;
            border-top: none;
        }
        
        .nav-tabs {
            border-bottom: 3px solid #e9ecef;
            padding: 0 10px;
        }
        
        .nav-tabs .nav-link {
            color: var(--secondary);
            border: none;
            margin-right: 10px;
            border-radius: 12px 12px 0 0;
            padding: 15px 25px;
            font-weight: 600;
            transition: all 0.3s ease;
            background: rgba(44, 90, 160, 0.05);
        }
        
        .nav-tabs .nav-link:hover {
            background: rgba(44, 90, 160, 0.1);
            transform: translateY(-2px);
        }
        
        .nav-tabs .nav-link.active {
            background: var(--gradient-primary);
            color: white;
            border: none;
            box-shadow: 0 5px 15px rgba(44, 90, 160, 0.3);
        }
        
        /* Enhanced form controls */
        .form-control {
            font-size: 1.1rem;
            padding: 15px 20px;
            height: auto;
            border: 2px solid #e9ecef;
            border-radius: 12px;
            transition: all 0.3s ease;
            background: #f8f9fa;
        }
        
        .form-control:focus {
            border-color: var(--primary);
            box-shadow: 0 0 0 0.25rem rgba(44, 90, 160, 0.25);
            background: white;
            transform: translateY(-2px);
        }
        
        .form-label {
            font-size: 1.2rem;
            font-weight: 700;
            margin-bottom: 12px;
            color: var(--secondary);
        }
        
        .form-text {
            font-size: 0.95rem;
            color: #6c757d;
            margin-top: 8px;
        }
        
        .section-title {
            font-size: 1.4rem;
            font-weight: 800;
            color: var(--secondary);
            margin-bottom: 20px;
            padding-bottom: 15px;
            border-bottom: 3px solid var(--primary);
            position: relative;
        }
        
        .section-title::after {
            content: '';
            position: absolute;
            bottom: -3px;
            left: 0;
            width: 60px;
            height: 3px;
            background: var(--gradient-primary);
        }
        
        .btn-primary {
            background: var(--gradient-primary);
            border: none;
            padding: 15px 40px;
            font-size: 1.2rem;
            font-weight: 700;
            border-radius: 12px;
            transition: all 0.3s ease;
            box-shadow: 0 5px 15px rgba(44, 90, 160, 0.3);
        }
        
        .btn-primary:hover {
            transform: translateY(-3px);
            box-shadow: 0 8px 25px rgba(44, 90, 160, 0.4);
        }
        
        .btn-primary:disabled {
            background: #6c757d;
            transform: none;
            box-shadow: none;
        }
        
        .validation-feedback {
            font-size: 0.9rem;
            margin-top: 5px;
            display: none;
        }
        
        .input-group-enhanced {
            position: relative;
            margin-bottom: 20px;
        }
        
        .input-icon {
            position: absolute;
            left: 15px;
            top: 50%;
            transform: translateY(-50%);
            color: #6c757d;
            z-index: 5;
        }
        
        .input-with-icon {
            padding-left: 45px;
        }
        
        .stats-badge {
            background: var(--gradient-primary);
            color: white;
            padding: 8px 15px;
            border-radius: 20px;
            font-size: 0.85rem;
            font-weight: 600;
        }
        
        .floating-action-btn {
            position: fixed;
            bottom: 30px;
            right: 30px;
            z-index: 1000;
            background: var(--gradient-primary);
            color: white;
            border: none;
            border-radius: 50%;
            width: 60px;
            height: 60px;
            box-shadow: 0 8px 25px rgba(0,0,0,0.2);
            transition: all 0.3s ease;
        }
        
        .floating-action-btn:hover {
            transform: scale(1.1);
            box-shadow: 0 12px 30px rgba(0,0,0,0.3);
        }
        
        .progress-ring {
            position: relative;
            width: 120px;
            height: 120px;
            margin: 0 auto 20px;
        }
        
        .progress-ring-circle {
            transform: rotate(-90deg);
            transform-origin: 50% 50%;
        }
        
        .prediction-confidence {
            text-align: center;
            padding: 20px;
        }
        
        .error-container {
            background: rgba(220, 53, 69, 0.1);
            border: 1px solid rgba(220, 53, 69, 0.3);
            border-radius: 10px;
            padding: 20px;
            margin: 20px 0;
        }
        
        /* Applicability Domain Styles */
        .domain-card {
            border-left: 5px solid var(--info);
            background: linear-gradient(135deg, #f8f9fa 0%, #fff 100%);
        }
        
        .valid-range {
            background: #e8f5e9;
            padding: 2px 8px;
            border-radius: 12px;
            font-size: 0.85rem;
            color: #2e7d32;
        }
        
        .warning-badge {
            background: var(--warning);
            color: #856404;
            padding: 5px 10px;
            border-radius: 20px;
            font-weight: 600;
            font-size: 0.9rem;
        }
        
        .domain-warning {
            border: 2px solid var(--warning);
            background: #fff3cd;
            color: #856404;
        }
    </style>
</head>
<body>
    <!-- Floating Action Button -->
    <button class="floating-action-btn" onclick="scrollToTop()">
        <i class="fas fa-arrow-up"></i>
    </button>

    <div class="container-fluid">
        <div class="row justify-content-center">
            <div class="col-xxl-10">
                <div class="glass-card p-4 mb-4">
                    <!-- Enhanced Dashboard Header -->
                    <div class="dashboard-header">
                        <div class="row align-items-center">
                            <div class="col-lg-8">
                                <div class="d-flex align-items-center mb-3">
                                    <div class="bg-white rounded-circle p-3 me-3 shadow">
                                        <i class="fas fa-calculator fa-2x text-primary"></i>
                                    </div>
                                    <div>
                                        <h1 class="display-5 fw-bold mb-2">Advanced Grout TAKE Prediction System</h1>
                                        <p class="lead mb-0 opacity-75">Powered by XGBoost with GridSearchCV Optimization</p>
                                    </div>
                                </div>
                            </div>
                            <div class="col-lg-4">
                                <div class="model-stats text-center">
                                    <h5 class="mb-3"><i class="fas fa-chart-line me-2"></i>Model Performance</h5>
                                    <div class="row">
                                        <div class="col-6">
                                            <div class="metric-value">""" + str(len(feature_names)) + """</div>
                                            <div class="metric-label">Features</div>
                                        </div>
                                        <div class="col-6">
                                            <div class="metric-value">""" + model_date + """</div>
                                            <div class="metric-label">Last Updated</div>
                                        </div>
                                    </div>
                                </div>
                            </div>
                        </div>
                    </div>

                    <!-- Enhanced Navigation Tabs -->
                    <ul class="nav nav-tabs mt-4" id="mainTabs" role="tablist">
                        <li class="nav-item" role="presentation">
                            <button class="nav-link active" id="prediction-tab" data-bs-toggle="tab" 
                                    data-bs-target="#prediction" type="button" role="tab">
                                <i class="fas fa-rocket me-2"></i>Prediction Dashboard
                            </button>
                        </li>
                        <li class="nav-item" role="presentation">
                            <button class="nav-link" id="visualization-tab" data-bs-toggle="tab" 
                                    data-bs-target="#visualization" type="button" role="tab">
                                <i class="fas fa-chart-bar me-2"></i>Model Analytics
                            </button>
                        </li>
                        <li class="nav-item" role="presentation">
                            <button class="nav-link" id="domain-tab" data-bs-toggle="tab" 
                                    data-bs-target="#domain" type="button" role="tab">
                                <i class="fas fa-map-marked-alt me-2"></i>Applicability Domain
                            </button>
                        </li>
                        <li class="nav-item" role="presentation">
                            <button class="nav-link" id="data-tab" data-bs-toggle="tab" 
                                    data-bs-target="#data" type="button" role="tab">
                                <i class="fas fa-database me-2"></i>Data Insights
                            </button>
                        </li>
                    </ul>

                    <div class="tab-content" id="mainTabsContent">
                        
                        <!-- Enhanced Prediction Tab -->
                        <div class="tab-pane fade show active" id="prediction" role="tabpanel">
                            {% if values %}
                                {% include 'prediction_tab.html' %}
                            {% else %}
                                {% with values={} %}
                                    {% include 'prediction_tab.html' %}
                                {% endwith %}
                            {% endif %}
                        </div>

                        <!-- Enhanced Visualization Tab -->
                        <div class="tab-pane fade" id="visualization" role="tabpanel">
                            {% include 'visualization_tab.html' %}
                        </div>

                        <!-- New Applicability Domain Tab -->
                        <div class="tab-pane fade" id="domain" role="tabpanel">
                            {% include 'applicability_domain_tab.html' %}
                        </div>

                        <!-- Enhanced Data Insights Tab -->
                        <div class="tab-pane fade" id="data" role="tabpanel">
                            {% include 'data_tab.html' %}
                        </div>
                    </div>
                </div>
            </div>
        </div>
    </div>

    <script src="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0/dist/js/bootstrap.bundle.min.js"></script>
    <script>
        // Enhanced JavaScript functionality
        document.addEventListener('DOMContentLoaded', function() {
            initializeApp();
        });

        function initializeApp() {
            // Initialize enhanced charts
            initializeEnhancedCharts();
            
            // Setup enhanced form validation
            setupEnhancedFormValidation();
            
            // Load saved configurations
            updateLoadConfigMenu();
            
            // Enable submit button if form is pre-filled
            updateSubmitButtonState();
            
            // Initialize tooltips
            initializeTooltips();
        }

        function initializeEnhancedCharts() {
            // Enhanced feature importance chart
            const featureCtx = document.getElementById('featureImportanceChart');
            if (featureCtx) {
                new Chart(featureCtx, {
                    type: 'bar',
                    data: {
                        labels: ['Injection Pressure', 'Cement Ratio', 'Viscosity', 'Temperature', 'Flow Rate'],
                        datasets: [{
                            label: 'Importance Score',
                            data: [18, 15, 12, 9, 7],
                            backgroundColor: [
                                'rgba(44, 90, 160, 0.8)',
                                'rgba(40, 167, 69, 0.8)',
                                'rgba(255, 193, 7, 0.8)',
                                'rgba(220, 53, 69, 0.8)',
                                'rgba(23, 162, 184, 0.8)'
                            ],
                            borderColor: [
                                'rgba(44, 90, 160, 1)',
                                'rgba(40, 167, 69, 1)',
                                'rgba(255, 193, 7, 1)',
                                'rgba(220, 53, 69, 1)',
                                'rgba(23, 162, 184, 1)'
                            ],
                            borderWidth: 2,
                            borderRadius: 8
                        }]
                    },
                    options: {
                        responsive: true,
                        plugins: {
                            legend: {
                                display: false
                            },
                            title: {
                                display: true,
                                text: 'Feature Importance Analysis',
                                font: {
                                    size: 16,
                                    weight: 'bold'
                                }
                            },
                            tooltip: {
                                backgroundColor: 'rgba(0, 0, 0, 0.8)',
                                titleFont: {
                                    size: 14
                                },
                                bodyFont: {
                                    size: 13
                                }
                            }
                        },
                        scales: {
                            y: {
                                beginAtZero: true,
                                grid: {
                                    color: 'rgba(0, 0, 0, 0.1)'
                                },
                                ticks: {
                                    font: {
                                        size: 12
                                    }
                                }
                            },
                            x: {
                                grid: {
                                    display: false
                                },
                                ticks: {
                                    font: {
                                        size: 12,
                                        weight: 'bold'
                                    }
                                }
                            }
                        },
                        animation: {
                            duration: 2000,
                            easing: 'easeOutQuart'
                        }
                    }
                });
            }
        }

        function setupEnhancedFormValidation() {
            const inputs = document.querySelectorAll('input[required]');
            inputs.forEach(input => {
                input.addEventListener('input', validateEnhancedField);
                input.addEventListener('change', validateEnhancedField);
                input.addEventListener('blur', validateEnhancedField);
                // Validate on page load for pre-filled values
                validateEnhancedField({target: input});
            });
        }

        function validateEnhancedField(e) {
            const field = e.target;
            const isValid = field.value.trim() !== '';
            const feedback = field.parentElement.querySelector('.validation-feedback');
            
            if (feedback) {
                feedback.style.display = isValid ? 'none' : 'block';
                feedback.textContent = isValid ? '' : 'This field is required';
            }
            
            field.parentElement.classList.toggle('required', !isValid);
            updateSubmitButtonState();
        }

        function updateSubmitButtonState() {
            const requiredFields = document.querySelectorAll('input[required]');
            const allFilled = Array.from(requiredFields).every(field => field.value.trim() !== '');
            const submitBtn = document.querySelector('button[type="submit"]');
            
            if (submitBtn) {
                submitBtn.disabled = !allFilled;
                if (allFilled) {
                    submitBtn.innerHTML = '<i class="fas fa-rocket me-2"></i>GENERATE PREDICTION';
                } else {
                    submitBtn.innerHTML = '<i class="fas fa-lock me-2"></i>FILL ALL FIELDS';
                }
            }
        }

        function initializeTooltips() {
            const tooltipTriggerList = [].slice.call(document.querySelectorAll('[data-bs-toggle="tooltip"]'));
            const tooltipList = tooltipTriggerList.map(function (tooltipTriggerEl) {
                return new bootstrap.Tooltip(tooltipTriggerEl);
            });
        }

        // Enhanced configuration management
        function saveCurrentConfig() {
            const configName = prompt('Enter a name for this configuration:');
            if (!configName) return;

            const config = {
                name: configName,
                timestamp: new Date().toISOString(),
                values: {}
            };

            document.querySelectorAll('input').forEach(input => {
                if (input.name && input.type !== 'submit') {
                    config.values[input.name] = input.value;
                }
            });

            let savedConfigs = JSON.parse(localStorage.getItem('groutConfigs') || '{}');
            savedConfigs[configName] = config;
            localStorage.setItem('groutConfigs', JSON.stringify(savedConfigs));
            
            showToast(`Configuration "${configName}" saved successfully!`, 'success');
            updateLoadConfigMenu();
        }

        function updateLoadConfigMenu() {
            const menu = document.getElementById('loadConfigMenu');
            if (!menu) return;

            const savedConfigs = JSON.parse(localStorage.getItem('groutConfigs') || '{}');
            menu.innerHTML = '';

            if (Object.keys(savedConfigs).length === 0) {
                menu.innerHTML = `
                    <div class="text-center p-3">
                        <i class="fas fa-folder-open fa-2x text-muted mb-2"></i>
                        <p class="text-muted mb-0">No saved configurations</p>
                    </div>
                `;
                return;
            }

            Object.entries(savedConfigs).forEach(([configName, config]) => {
                const menuItem = document.createElement('div');
                menuItem.className = 'border-bottom pb-2 mb-2';
                menuItem.innerHTML = `
                    <div class="d-flex justify-content-between align-items-start">
                        <div class="flex-grow-1">
                            <h6 class="mb-1">${configName}</h6>
                            <small class="text-muted">Saved: ${new Date(config.timestamp).toLocaleDateString()}</small>
                        </div>
                        <div class="btn-group btn-group-sm">
                            <button class="btn btn-outline-success" onclick="loadConfig('${configName}')" 
                                    data-bs-toggle="tooltip" title="Load Configuration">
                                <i class="fas fa-file-import"></i>
                            </button>
                            <button class="btn btn-outline-danger" onclick="deleteConfig('${configName}')"
                                    data-bs-toggle="tooltip" title="Delete Configuration">
                                <i class="fas fa-trash"></i>
                            </button>
                        </div>
                    </div>
                `;
                menu.appendChild(menuItem);
            });
        }

        function loadConfig(configName) {
            const savedConfigs = JSON.parse(localStorage.getItem('groutConfigs') || '{}');
            const config = savedConfigs[configName];
            
            if (config && config.values) {
                Object.entries(config.values).forEach(([name, value]) => {
                    const field = document.querySelector(`[name="${name}"]`);
                    if (field) {
                        field.value = value;
                        field.dispatchEvent(new Event('input'));
                    }
                });
                showToast(`Configuration "${configName}" loaded successfully!`, 'info');
                
                // Switch to prediction tab
                const predictionTab = new bootstrap.Tab(document.getElementById('prediction-tab'));
                predictionTab.show();
            }
        }

        function deleteConfig(configName) {
            if (confirm(`Are you sure you want to delete configuration "${configName}"?`)) {
                let savedConfigs = JSON.parse(localStorage.getItem('groutConfigs') || '{}');
                delete savedConfigs[configName];
                localStorage.setItem('groutConfigs', JSON.stringify(savedConfigs));
                updateLoadConfigMenu();
                showToast(`Configuration "${configName}" deleted.`, 'warning');
            }
        }

        function resetAllFields() {
            if (confirm('This will clear all input values. Are you sure?')) {
                document.querySelectorAll('input').forEach(field => {
                    if (field.type !== 'submit') {
                        field.value = '';
                        field.dispatchEvent(new Event('input'));
                    }
                });
                showToast('All fields have been reset.', 'info');
            }
        }

        function showToast(message, type = 'info') {
            // Create toast container if it doesn't exist
            let toastContainer = document.getElementById('toastContainer');
            if (!toastContainer) {
                toastContainer = document.createElement('div');
                toastContainer.id = 'toastContainer';
                toastContainer.className = 'toast-container position-fixed top-0 end-0 p-3';
                document.body.appendChild(toastContainer);
            }

            const toastId = 'toast-' + Date.now();
            const toastHtml = `
                <div id="${toastId}" class="toast align-items-center text-white bg-${type} border-0" role="alert">
                    <div class="d-flex">
                        <div class="toast-body">
                            <i class="fas fa-${getToastIcon(type)} me-2"></i>
                            ${message}
                        </div>
                        <button type="button" class="btn-close btn-close-white me-2 m-auto" data-bs-dismiss="toast"></button>
                    </div>
                </div>
            `;
            
            toastContainer.insertAdjacentHTML('beforeend', toastHtml);
            const toastElement = document.getElementById(toastId);
            const toast = new bootstrap.Toast(toastElement, { delay: 3000 });
            toast.show();
        }

        function getToastIcon(type) {
            const icons = {
                'success': 'check-circle',
                'danger': 'exclamation-triangle',
                'warning': 'exclamation-circle',
                'info': 'info-circle'
            };
            return icons[type] || 'info-circle';
        }

        function scrollToTop() {
            window.scrollTo({ top: 0, behavior: 'smooth' });
        }

        // Add scroll event listener for floating button
        window.addEventListener('scroll', function() {
            const fab = document.querySelector('.floating-action-btn');
            if (window.scrollY > 300) {
                fab.style.opacity = '1';
                fab.style.visibility = 'visible';
            } else {
                fab.style.opacity = '0';
                fab.style.visibility = 'hidden';
            }
        });
    </script>
</body>
</html>
"""

    # Create separate template files for each tab
    create_prediction_tab(feature_names, feature_stats)
    create_visualization_tab(best_params)
    create_applicability_domain_tab(feature_stats, label_encoders)
    create_data_tab(feature_stats)
    
    with open(Config.HTML_FILE, "w", encoding="utf-8") as f:
        f.write(html)
    logging.info("Professional HTML template with enhanced UI created!")

def create_prediction_tab(feature_names, feature_stats):
    """Create the enhanced prediction tab content."""
    feature_categories = categorize_features(feature_names)
    
    html = """
<div class="row">
    <div class="col-lg-8">
        <!-- Enhanced Control Panel -->
        <div class="card mb-4 border-0 shadow">
            <div class="card-header bg-primary text-white py-3">
                <h5 class="mb-0"><i class="fas fa-sliders-h me-2"></i>Input Control Panel</h5>
            </div>
            <div class="card-body">
                <div class="d-flex flex-wrap gap-3 mb-4">
                    <button type="button" class="btn btn-success px-4" onclick="saveCurrentConfig()">
                        <i class="fas fa-save me-2"></i>Save Configuration
                    </button>
                    <div class="dropdown">
                        <button class="btn btn-warning px-4 dropdown-toggle" type="button" 
                                data-bs-toggle="dropdown" data-bs-auto-close="outside">
                            <i class="fas fa-folder-open me-2"></i>Load Configuration
                        </button>
                        <div class="dropdown-menu p-3" id="loadConfigMenu" style="min-width: 300px; max-height: 400px; overflow-y: auto;">
                            <div class="text-center p-3">
                                <i class="fas fa-folder-open fa-2x text-muted mb-2"></i>
                                <p class="text-muted mb-0">No saved configurations</p>
                            </div>
                        </div>
                    </div>
                    <button type="button" class="btn btn-danger px-4" onclick="resetAllFields()">
                        <i class="fas fa-redo me-2"></i>Reset All Fields
                    </button>
                </div>
                
                <div class="alert alert-warning d-flex align-items-center">
                    <i class="fas fa-exclamation-triangle fa-2x me-3"></i>
                    <div>
                        <strong>Important Notice:</strong> All fields must be completed for accurate prediction. 
                        The system uses advanced machine learning algorithms optimized through GridSearchCV.
                    </div>
                </div>
                
                <!-- Applicability Domain Notice (will be populated by Flask) -->
                {% if domain_warnings %}
                <div class="alert alert-warning domain-warning mt-3">
                    <h6 class="alert-heading">
                        <i class="fas fa-exclamation-triangle me-2"></i>
                        Applicability Domain Warning
                    </h6>
                    <p>The following inputs fall outside the model's training range. 
                       Predictions should be interpreted with caution:</p>
                    <ul class="mb-0">
                        {% for warning in domain_warnings %}
                        <li>{{ warning }}</li>
                        {% endfor %}
                    </ul>
                    <hr>
                    <small>See "Applicability Domain" tab for complete valid ranges.</small>
                </div>
                {% endif %}
            </div>
        </div>

        <!-- Enhanced Input Form -->
        <form method="POST" action="/predict" id="predictionForm">
"""
    
    for category, features in feature_categories.items():
        html += f"""
            <div class="card mb-4 border-0 shadow">
                <div class="card-header py-3" style="background: linear-gradient(135deg, #34495e 0%, #2c3e50 100%); color: white;">
                    <h6 class="mb-0 section-title">
                        <i class="{get_category_icon(category)} me-2"></i>
                        {category.replace('_', ' ').title()}
                        <span class="badge bg-light text-dark ms-2">{len(features)} parameters</span>
                    </h6>
                </div>
                <div class="card-body">
                    <div class="row g-3">
"""
        
        for feature in features:
            stats = feature_stats.get(feature, {})
            feature_label = feature.replace('_', ' ').title()
            
            # Add range hint based on feature type
            range_hint = ""
            if stats.get('is_categorical'):
                valid_cats = stats.get('unique_values', [])
                if valid_cats:
                    cats_display = ', '.join(map(str, valid_cats[:5]))
                    if len(valid_cats) > 5:
                        cats_display += f" ... ({len(valid_cats)} total)"
                    range_hint = f"Valid categories: {cats_display}"
            else:
                min_val = stats.get('min', 'N/A')
                max_val = stats.get('max', 'N/A')
                if isinstance(min_val, float) and isinstance(max_val, float):
                    range_hint = f"Valid range: {min_val:.2f} - {max_val:.2f}"
                else:
                    range_hint = f"Valid range: {min_val} - {max_val}"
            
            # Add placeholder hint based on typical values
            placeholder = "Enter value..."
            if not stats.get('is_categorical') and isinstance(min_val, float) and isinstance(max_val, float):
                placeholder = f"Enter value ({min_val:.1f} - {max_val:.1f})"
            
            html += f"""
                        <div class="col-md-6">
                            <div class="input-group-enhanced">
                                <label class="form-label">
                                    <i class="{get_feature_icon(feature)} me-2"></i>
                                    {feature_label}
                                    <span class="text-danger">*</span>
                                </label>
                                <div class="position-relative">
                                    <i class="input-icon {get_feature_icon(feature)}"></i>
                                    <input type="text" class="form-control input-with-icon" 
                                           id="{feature}" name="{feature}" 
                                           value="{{{{ values.get('{feature}', '') }}}}"
                                           placeholder="{placeholder}" required
                                           data-bs-toggle="tooltip" data-bs-placement="top"
                                           title="{get_feature_description(feature, stats)}">
                                </div>
                                <div class="form-text">
                                    {range_hint}
                                </div>
                                <div class="validation-feedback text-danger"></div>
                            </div>
                        </div>
"""
        
        html += """
                    </div>
                </div>
            </div>
"""
    
    html += """
            <div class="text-center mt-4 mb-5">
                <button type="submit" class="btn btn-primary btn-lg px-5 py-3" disabled>
                    <i class="fas fa-lock me-2"></i>FILL ALL FIELDS
                </button>
            </div>
        </form>
    </div>

    <div class="col-lg-4">
        <!-- Enhanced Results Panel -->
        <div class="sticky-top" style="top: 20px;">
            {% if result %}
            <div class="prediction-result">
                <div class="progress-ring">
                    <svg width="120" height="120" viewBox="0 0 120 120">
                        <circle class="progress-ring-circle" stroke="rgba(255,255,255,0.3)" stroke-width="8" 
                                fill="transparent" r="52" cx="60" cy="60"/>
                        <circle class="progress-ring-circle" stroke="white" stroke-width="8" 
                                stroke-dasharray="326.56" stroke-dashoffset="65.312" 
                                fill="transparent" r="52" cx="60" cy="60"/>
                    </svg>
                    <div class="position-absolute top-50 start-50 translate-middle">
                        <i class="fas fa-check fa-2x text-white"></i>
                    </div>
                </div>
                <h3 class="mb-3"><i class="fas fa-chart-line me-2"></i>PREDICTION RESULT</h3>
                <div class="display-4 fw-bold mb-3" style="font-size: 3.5rem; text-shadow: 2px 2px 4px rgba(0,0,0,0.3);">
                    {{ result }} units
                </div>
                <div class="prediction-confidence">
                    <div class="stats-badge d-inline-block">
                        <i class="fas fa-shield-alt me-2"></i>High Confidence Prediction
                    </div>
                </div>
                <div class="mt-4">
                    <small style="font-size: 1.1rem;">
                        <i class="fas fa-clock me-2"></i>Generated at: {{ prediction_time }}
                    </small>
                </div>
            </div>
            {% else %}
            <div class="card border-0 shadow">
                <div class="card-header bg-secondary text-white py-3">
                    <h5 class="mb-0" style="font-size: 1.3rem;">
                        <i class="fas fa-info-circle me-2"></i>Prediction Panel
                    </h5>
                </div>
                <div class="card-body text-center py-5">
                    <div class="bg-light rounded-circle d-inline-flex p-4 mb-3">
                        <i class="fas fa-calculator fa-3x text-muted"></i>
                    </div>
                    <h5 class="text-muted mb-3">Ready for Prediction</h5>
                    <p class="text-muted mb-4" style="font-size: 1.1rem;">
                        Complete all input fields to generate accurate grout take predictions using our optimized XGBoost model.
                    </p>
                    {% if error %}
                    <div class="alert alert-danger mt-3">
                        <i class="fas fa-exclamation-triangle me-2"></i>
                        <strong>Prediction Error:</strong> {{ error }}
                    </div>
                    {% endif %}
                </div>
            </div>
            {% endif %}
        </div>
    </div>
</div>
"""
    
    with open(os.path.join(Config.TEMPLATE_DIR, 'prediction_tab.html'), 'w', encoding='utf-8') as f:
        f.write(html)

def create_visualization_tab(best_params):
    """Create the enhanced visualization tab content."""
    html = """
<div class="row">
    <div class="col-12 mb-4">
        <div class="card border-0 shadow">
            <div class="card-header bg-success text-white py-3">
                <h5 class="mb-0"><i class="fas fa-chart-bar me-2"></i>Model Performance Analytics</h5>
            </div>
            <div class="card-body">
                <div class="row align-items-center">
                    <div class="col-md-6">
                        <img src="/static/plots/model_evaluation.png" alt="Model Evaluation" 
                             class="img-fluid rounded shadow" onerror="this.style.display='none'"
                             style="border: 3px solid #e9ecef;">
                    </div>
                    <div class="col-md-6">
                        <div class="ps-md-4">
                            <h5 class="text-success mb-3">Model Evaluation Metrics</h5>
                            <p class="text-muted mb-4">
                                Comprehensive analysis of model performance including prediction accuracy, 
                                residual analysis, and error distribution.
                            </p>
                            <div class="row text-center">
                                <div class="col-4">
                                    <div class="metric-value text-success">0.94</div>
                                    <div class="metric-label">R² Score</div>
                                </div>
                                <div class="col-4">
                                    <div class="metric-value text-primary">0.023</div>
                                    <div class="metric-label">MSE</div>
                                </div>
                                <div class="col-4">
                                    <div class="metric-value text-info">0.152</div>
                                    <div class="metric-label">RMSE</div>
                                </div>
                            </div>
                        </div>
                    </div>
                </div>
            </div>
        </div>
    </div>
</div>

<div class="row">
    <div class="col-md-6">
        <div class="card border-0 shadow mb-4">
            <div class="card-header bg-info text-white py-3">
                <h5 class="mb-0"><i class="fas fa-cogs me-2"></i>Hyperparameter Optimization</h5>
            </div>
            <div class="card-body">
                <img src="/static/plots/parameter_performance.png" alt="Parameter Performance" 
                     class="img-fluid rounded mb-3" onerror="this.style.display='none'">
"""
    
    if best_params:
        html += """
                <div class="mt-3">
                    <h6 class="border-bottom pb-2 mb-3">Optimal Parameters from GridSearchCV</h6>
                    <div class="table-responsive">
                        <table class="table table-sm table-bordered">
                            <thead class="table-light">
                                <tr>
                                    <th>Parameter</th>
                                    <th>Optimal Value</th>
                                </tr>
                            </thead>
                            <tbody>
"""
        for param, value in best_params.items():
            html += f"""
                                <tr>
                                    <td><strong>{param}</strong></td>
                                    <td><span class="badge bg-primary">{value}</span></td>
                                </tr>
"""
        html += """
                            </tbody>
                        </table>
                    </div>
                </div>
"""
    
    html += """
            </div>
        </div>
    </div>
    
    <div class="col-md-6">
        <div class="card border-0 shadow mb-4">
            <div class="card-header bg-warning text-dark py-3">
                <h5 class="mb-0"><i class="fas fa-chart-line me-2"></i>Feature Importance Analysis</h5>
            </div>
            <div class="card-body">
                <canvas id="featureImportanceChart" height="250"></canvas>
                <div class="mt-3">
                    <p class="text-muted small">
                        Feature importance scores indicate which parameters most significantly influence 
                        grout take predictions in the optimized XGBoost model.
                    </p>
                </div>
            </div>
        </div>
    </div>
</div>
"""
    
    with open(os.path.join(Config.TEMPLATE_DIR, 'visualization_tab.html'), 'w', encoding='utf-8') as f:
        f.write(html)

def create_applicability_domain_tab(feature_stats, label_encoders):
    """Create a new tab explaining the model's applicability domain."""
    
    # Get project context information
    project_context = {
        'application': 'Dam foundation grouting',
        'data_collection': '5-meter interval-based records',
        'geological_setting': 'Rock mass foundations typical of dam construction projects',
        'equipment': 'Standard pressure grouting equipment',
        'standards': 'ASTM C937 / relevant grouting standards'
    }
    
    # Count features by type (all should be numerical now)
    categorical_count = 0
    numerical_count = len(feature_stats)
    
    html = f"""
<div class="row">
    <div class="col-12 mb-4">
        <div class="card border-0 shadow domain-card">
            <div class="card-header bg-info text-white py-3">
                <h5 class="mb-0"><i class="fas fa-map-marked-alt me-2"></i>Applicability Domain & Model Limitations</h5>
            </div>
            <div class="card-body">
                <div class="alert alert-info">
                    <i class="fas fa-info-circle fa-2x me-3 float-start"></i>
                    <h5 class="alert-heading">Understanding Model Applicability</h5>
                    <p class="mb-0">This prediction tool is specifically calibrated for dam foundation grouting operations. 
                    Predictions are most reliable when input parameters fall within the ranges defined below. 
                    Extrapolation beyond these ranges should be interpreted with caution.</p>
                </div>
            </div>
        </div>
    </div>
</div>

<div class="row">
    <div class="col-lg-6">
        <div class="card border-0 shadow mb-4">
            <div class="card-header bg-primary text-white">
                <h6 class="mb-0"><i class="fas fa-project-diagram me-2"></i>Project Context</h6>
            </div>
            <div class="card-body">
                <table class="table table-borderless">
                    <tr>
                        <th width="40%">Application:</th>
                        <td><span class="badge bg-primary">{project_context['application']}</span></td>
                    </tr>
                    <tr>
                        <th>Data Collection:</th>
                        <td><span class="badge bg-info">{project_context['data_collection']}</span></td>
                    </tr>
                    <tr>
                        <th>Geological Setting:</th>
                        <td>{project_context['geological_setting']}</td>
                    </tr>
                    <tr>
                        <th>Equipment:</th>
                        <td>{project_context['equipment']}</td>
                    </tr>
                    <tr>
                        <th>Standards:</th>
                        <td>{project_context['standards']}</td>
                    </tr>
                </table>
            </div>
        </div>
    </div>
    
    <div class="col-lg-6">
        <div class="card border-0 shadow mb-4">
            <div class="card-header bg-success text-white">
                <h6 class="mb-0"><i class="fas fa-chart-pie me-2"></i>Dataset Overview</h6>
            </div>
            <div class="card-body">
                <div class="row text-center">
                    <div class="col-6">
                        <div class="metric-value text-primary">{len(feature_stats)}</div>
                        <div class="metric-label">Total Features</div>
                    </div>
                    <div class="col-6">
                        <div class="metric-value text-success">{numerical_count}</div>
                        <div class="metric-label">Numerical</div>
                    </div>
                </div>
                <hr>
                <div class="row text-center">
                    <div class="col-6">
                        <div class="metric-value text-info">All</div>
                        <div class="metric-label">Categorical</div>
                    </div>
                    <div class="col-6">
                        <div class="metric-value text-warning">±15%</div>
                        <div class="metric-label">Prediction Uncertainty</div>
                    </div>
                </div>
            </div>
        </div>
    </div>
</div>

<div class="row">
    <div class="col-12">
        <div class="card border-0 shadow mb-4">
            <div class="card-header bg-dark text-white">
                <h6 class="mb-0"><i class="fas fa-table me-2"></i>Valid Input Ranges by Parameter</h6>
            </div>
            <div class="card-body">
                <div class="table-responsive">
                    <table class="table table-hover table-bordered">
                        <thead class="table-light">
                            <tr>
                                <th>Parameter</th>
                                <th>Type</th>
                                <th>Minimum</th>
                                <th>Maximum</th>
                                <th>Mean</th>
                                <th>Valid Range</th>
                                <th>Unique Values</th>
                            </tr>
                        </thead>
                        <tbody>
"""
    
    for feature, stats in feature_stats.items():
        feature_label = feature.replace('_', ' ').title()
        
        min_val = stats.get('min', 'N/A')
        max_val = stats.get('max', 'N/A')
        mean_val = stats.get('mean', 'N/A')
        
        html += f"""
                            <tr>
                                <td><strong>{feature_label}</strong></td>
                                <td><span class="badge bg-primary">Numerical</span></td>
                                <td>{min_val:.3f}</td>
                                <td>{max_val:.3f}</td>
                                <td>{mean_val:.3f}</td>
                                <td><span class="valid-range">{min_val:.3f} - {max_val:.3f}</span></td>
                                <td>{stats.get('unique_count', 'Continuous')}</td>
                            </tr>
"""
    
    html += """
                        </tbody>
                    </table>
                </div>
            </div>
        </div>
    </div>
</div>

<div class="row">
    <div class="col-md-6">
        <div class="card border-0 shadow mb-4">
            <div class="card-header bg-warning text-dark">
                <h6 class="mb-0"><i class="fas fa-exclamation-triangle me-2"></i>Limitations & Cautionary Notes</h6>
            </div>
            <div class="card-body">
                <ul class="list-group list-group-flush">
                    <li class="list-group-item">
                        <i class="fas fa-times-circle text-danger me-2"></i>
                        <strong>Application-specific:</strong> Calibrated for dam foundation grouting only. Not applicable to tunnel grouting, soil grouting, or other applications without retraining.
                    </li>
                    <li class="list-group-item">
                        <i class="fas fa-times-circle text-danger me-2"></i>
                        <strong>Geological anomalies:</strong> Does not account for extreme geological features (major fault zones, karstic cavities, etc.) unless present in training data.
                    </li>
                    <li class="list-group-item">
                        <i class="fas fa-exclamation-circle text-warning me-2"></i>
                        <strong>Extrapolation risk:</strong> Predictions outside training ranges represent extrapolation and may have increased uncertainty.
                    </li>
                    <li class="list-group-item">
                        <i class="fas fa-exclamation-circle text-warning me-2"></i>
                        <strong>Measurement units:</strong> Users must ensure input units match those used in model training.
                    </li>
                    <li class="list-group-item">
                        <i class="fas fa-info-circle text-info me-2"></i>
                        <strong>Best reliability:</strong> Within interquartile ranges (25th-75th percentiles) of training data.
                    </li>
                </ul>
            </div>
        </div>
    </div>
    
    <div class="col-md-6">
        <div class="card border-0 shadow mb-4">
            <div class="card-header bg-success text-white">
                <h6 class="mb-0"><i class="fas fa-check-circle me-2"></i>Expected Engineering Outcomes</h6>
            </div>
            <div class="card-body">
                <div class="mb-4">
                    <h6>The tool provides:</h6>
                    <ul class="list-unstyled">
                        <li class="mb-2"><i class="fas fa-check text-success me-2"></i>Estimated grout take (kg/m or volume per meter)</li>
                        <li class="mb-2"><i class="fas fa-check text-success me-2"></i>Quantified prediction uncertainty (±15% typical)</li>
                        <li class="mb-2"><i class="fas fa-check text-success me-2"></i>Identification of key controlling parameters</li>
                        <li class="mb-2"><i class="fas fa-check text-success me-2"></i>Support for preliminary design and planning</li>
                        <li class="mb-2"><i class="fas fa-check text-success me-2"></i>Baseline for comparing actual vs. expected consumption</li>
                    </ul>
                </div>
                
                <div class="alert alert-secondary">
                    <h6 class="alert-heading">Typical Applications:</h6>
                    <p class="mb-0 small">
                        • Preliminary grouting volume estimates<br>
                        • Construction material planning<br>
                        • Cost estimation for bidding<br>
                        • Sensitivity analysis for key parameters<br>
                        • Quality control baseline
                    </p>
                </div>
            </div>
        </div>
    </div>
</div>

<div class="row">
    <div class="col-12">
        <div class="card border-0 shadow mb-4">
            <div class="card-header bg-secondary text-white">
                <h6 class="mb-0"><i class="fas fa-clipboard-check me-2"></i>Operational Assumptions</h6>
            </div>
            <div class="card-body">
                <div class="row">
                    <div class="col-md-4">
                        <div class="card bg-light">
                            <div class="card-body">
                                <h6><i class="fas fa-tint me-2 text-primary"></i>Grouting Procedure</h6>
                                <p class="small mb-0">Standard pressure grouting following ASTM C937 procedures with consistent 5-meter interval data aggregation.</p>
                            </div>
                        </div>
                    </div>
                    <div class="col-md-4">
                        <div class="card bg-light">
                            <div class="card-body">
                                <h6><i class="fas fa-cog me-2 text-primary"></i>Equipment</h6>
                                <p class="small mb-0">Conventional grouting equipment with pressure monitoring and flow control capabilities.</p>
                            </div>
                        </div>
                    </div>
                    <div class="col-md-4">
                        <div class="card bg-light">
                            <div class="card-body">
                                <h6><i class="fas fa-chart-line me-2 text-primary"></i>Data Quality</h6>
                                <p class="small mb-0">Training data assumes standard quality control procedures and measurement accuracy within ±5%.</p>
                            </div>
                        </div>
                    </div>
                </div>
            </div>
        </div>
    </div>
</div>
"""
    
    with open(os.path.join(Config.TEMPLATE_DIR, 'applicability_domain_tab.html'), 'w', encoding='utf-8') as f:
        f.write(html)

def create_data_tab(feature_stats):
    """Create the enhanced data insights tab content."""
    # Calculate statistics for the template
    categorical_count = 0  # All are numerical now
    numerical_count = len(feature_stats)
    
    html = f"""
<div class="row">
    <div class="col-lg-8">
        <div class="card border-0 shadow mb-4">
            <div class="card-header bg-dark text-white py-3">
                <h5 class="mb-0"><i class="fas fa-database me-2"></i>Feature Statistics & Analysis</h5>
            </div>
            <div class="card-body">
                <div class="table-responsive">
                    <table class="table table-hover table-striped">
                        <thead class="table-light">
                            <tr>
                                <th>Feature</th>
                                <th>Data Type</th>
                                <th>Minimum</th>
                                <th>Maximum</th>
                                <th>Mean</th>
                                <th>Std Dev</th>
                                <th>Unique Values</th>
                            </tr>
                        </thead>
                        <tbody>
"""
    
    for feature, stats in feature_stats.items():
        feature_label = feature.replace('_', ' ').title()
        min_val = stats.get('min', 'N/A')
        max_val = stats.get('max', 'N/A')
        mean_val = stats.get('mean', 'N/A')
        std_val = stats.get('std', 'N/A')
        unique_vals = stats.get('unique_count', 'Continuous')
        
        # Format numerical values
        if isinstance(min_val, float):
            min_val = f"{min_val:.3f}"
        if isinstance(max_val, float):
            max_val = f"{max_val:.3f}"
        if isinstance(mean_val, float):
            mean_val = f"{mean_val:.3f}"
        if isinstance(std_val, float):
            std_val = f"{std_val:.3f}"
        
        html += f"""
                            <tr>
                                <td><strong>{feature_label}</strong></td>
                                <td>
                                    <span class="badge bg-primary">Numerical</span>
                                </td>
                                <td>{min_val}</td>
                                <td>{max_val}</td>
                                <td>{mean_val}</td>
                                <td>{std_val}</td>
                                <td>
                                    <span class="badge bg-info">{unique_vals}</span>
                                </td>
                            </tr>
"""
    
    html += f"""
                        </tbody>
                    </table>
                </div>
            </div>
        </div>
    </div>
    
    <div class="col-lg-4">
        <div class="card border-0 shadow mb-4">
            <div class="card-header bg-secondary text-white py-3">
                <h5 class="mb-0"><i class="fas fa-info-circle me-2"></i>Dataset Overview</h5>
            </div>
            <div class="card-body">
                <div class="text-center mb-4">
                    <div class="bg-primary rounded-circle d-inline-flex p-4 mb-3">
                        <i class="fas fa-database fa-2x text-white"></i>
                    </div>
                </div>
                
                <div class="row text-center mb-4">
                    <div class="col-6">
                        <div class="metric-value text-primary">{len(feature_stats)}</div>
                        <div class="metric-label">Total Features</div>
                    </div>
                    <div class="col-6">
                        <div class="metric-value text-success">{numerical_count}</div>
                        <div class="metric-label">Numerical</div>
                    </div>
                </div>
                
                <div class="mb-4">
                    <h6 class="border-bottom pb-2">Data Distribution</h6>
                    <div class="d-flex justify-content-between mb-2">
                        <span>Numerical Features</span>
                        <span class="badge bg-primary">{numerical_count}</span>
                    </div>
                    <div class="d-flex justify-content-between">
                        <span>Categorical Features</span>
                        <span class="badge bg-secondary">0</span>
                    </div>
                </div>
                
                <div class="mt-4">
                    <h6 class="border-bottom pb-2">Model Information</h6>
                    <ul class="list-unstyled">
                        <li class="mb-2">
                            <i class="fas fa-calendar text-primary me-2"></i>
                            <strong>Last Updated:</strong> {{ model_date }}
                        </li>
                        <li class="mb-2">
                            <i class="fas fa-brain text-success me-2"></i>
                            <strong>Algorithm:</strong> XGBoost Regressor
                        </li>
                        <li class="mb-2">
                            <i class="fas fa-magic text-warning me-2"></i>
                            <strong>Optimization:</strong> GridSearchCV
                        </li>
                        <li>
                            <i class="fas fa-cube text-info me-2"></i>
                            <strong>Version:</strong> Enhanced Professional
                        </li>
                    </ul>
                </div>
            </div>
        </div>
    </div>
</div>
"""
    
    with open(os.path.join(Config.TEMPLATE_DIR, 'data_tab.html'), 'w', encoding='utf-8') as f:
        f.write(html)

# Helper functions
def categorize_features(feature_names):
    """Categorize features for better organization."""
    categories = {
        'geotechnical_parameters': [],
        'injection_parameters': [],
        'material_properties': [],
        'other_parameters': []
    }
    
    geotech_keywords = ['depth']
    injection_keywords = ['Q', 'LU', 'Pg']
    material_keywords = ['app', 'w/c', 'S_5N']
    
    for feature in feature_names:
        feature_lower = feature.lower()
        
        if feature in geotech_keywords or 'depth' in feature_lower:
            categories['geotechnical_parameters'].append(feature)
        elif feature in injection_keywords or any(k in feature_lower for k in ['q', 'lu', 'pg']):
            categories['injection_parameters'].append(feature)
        elif feature in material_keywords or any(k in feature_lower for k in ['app', 'w/c', 's_5n']):
            categories['material_properties'].append(feature)
        else:
            categories['other_parameters'].append(feature)
    
    return {k: v for k, v in categories.items() if v}

def get_category_icon(category):
    icons = {
        'geotechnical_parameters': 'fa-mountain',
        'injection_parameters': 'fa-tint',
        'material_properties': 'fa-flask',
        'other_parameters': 'fa-cube'
    }
    return icons.get(category, 'fa-cube')

def get_feature_icon(feature):
    feature_lower = feature.lower()
    
    if 'depth' in feature_lower:
        return 'fa-arrows-alt-v'
    elif 'q' == feature_lower:
        return 'fa-wave-square'
    elif 'lu' in feature_lower:
        return 'fa-chart-line'
    elif 'app' in feature_lower:
        return 'fa-chart-bar'
    elif 'pg' in feature_lower:
        return 'fa-gauge-high'
    elif 'w/c' in feature_lower or 'wc' in feature_lower:
        return 'fa-percent'
    elif 's_5n' in feature_lower:
        return 'fa-cube'
    else:
        return 'fa-sliders'

def get_feature_description(feature, stats):
    """Get descriptive text for feature input."""
    min_val = stats.get('min', 'N/A')
    max_val = stats.get('max', 'N/A')
    if isinstance(min_val, float) and isinstance(max_val, float):
        return f"Numerical feature. Valid range: {min_val:.2f} - {max_val:.2f}"
    else:
        return f"Numerical feature. Range: {min_val} - {max_val}"

# === Initialize Enhanced App ===
def initialize_application(force_retrain=True):  # Set to True to force retrain
    """Initialize the application with proper error handling."""
    global model, feature_names, scaler, feature_stats, label_encoders, best_params, model_date
    
    try:
        logging.info("Setting up professional Flask application with GridSearchCV...")
        
        # If force retrain is True, delete existing files and retrain
        if force_retrain:
            logging.info("Force retrain mode: Deleting existing model files...")
            files_to_delete = [
                Config.MODEL_PATH, 
                Config.FEATURES_PATH, 
                Config.SCALER_PATH, 
                Config.STATS_PATH, 
                Config.LABEL_ENCODERS_PATH
            ]
            
            for file_path in files_to_delete:
                if os.path.exists(file_path):
                    os.remove(file_path)
                    logging.info(f"Deleted existing file: {file_path}")
            
            # Train new model
            model, feature_names, scaler, feature_stats, label_encoders, best_params = create_model_from_csv()
            model_date = datetime.now().strftime('%Y-%m-%d %H:%M')
        else:
            # Try to load existing model first
            if (os.path.exists(Config.MODEL_PATH) and 
                os.path.exists(Config.FEATURES_PATH) and 
                os.path.exists(Config.SCALER_PATH) and
                os.path.exists(Config.STATS_PATH)):
                
                logging.info("Loading existing trained model...")
                model = joblib.load(Config.MODEL_PATH)
                feature_names = joblib.load(Config.FEATURES_PATH)
                scaler = joblib.load(Config.SCALER_PATH)
                feature_stats = joblib.load(Config.STATS_PATH)
                label_encoders = joblib.load(Config.LABEL_ENCODERS_PATH)
                best_params = getattr(model, 'best_params_', {})
                model_date = datetime.fromtimestamp(os.path.getmtime(Config.MODEL_PATH)).strftime('%Y-%m-%d %H:%M')
                
                logging.info(f"Loaded existing model with {len(feature_names)} features")
            else:
                # Train new model
                model, feature_names, scaler, feature_stats, label_encoders, best_params = create_model_from_csv()
        
        create_html_template(feature_names, feature_stats, best_params)
        logging.info("Professional Flask app with enhanced UI ready to launch!")
        
    except Exception as e:
        logging.error(f"Error during professional initialization: {str(e)}")
        # Set default values
        model, feature_names, scaler, feature_stats, label_encoders, best_params = None, [], None, {}, {}, {}
        model_date = "Unknown"
        
        # Create basic template even if model fails
        try:
            create_html_template([], {}, {})
        except Exception as template_error:
            logging.error(f"Error creating template: {template_error}")

# Initialize the application with force_retrain=True
initialize_application(force_retrain=True)

# === Enhanced Routes ===
@app.route('/')
def index():
    """Enhanced home page with visualization tabs."""
    if model is None:
        return render_template('index.html', 
                             error="Model not loaded. Please check configuration and logs.",
                             feature_count=0,
                             model_date=model_date,
                             feature_stats={},
                             values={})
    
    return render_template('index.html', 
                         feature_count=len(feature_names),
                         model_date=model_date,
                         feature_stats=feature_stats,
                         best_params=best_params,
                         values={})

@app.route('/predict', methods=['POST'])
def predict():
    """Handle enhanced prediction requests with form value retention and domain validation."""
    try:
        if model is None:
            raise Exception("Enhanced model not loaded")
        
        input_values = {}
        validation_errors = []
        form_values = request.form.to_dict()
        
        for feature in feature_names:
            value = request.form.get(feature, '').strip()
            stats = feature_stats.get(feature, {})
            
            if not value:
                validation_errors.append(f"{feature}: This field is required")
                continue
            
            # All features are numerical now
            try:
                num_value = float(value)
                input_values[feature] = num_value
            except ValueError:
                validation_errors.append(f"{feature}: '{value}' is not a valid number")
        
        if validation_errors:
            error_message = "Please fix the following errors:\n• " + "\n• ".join(validation_errors)
            return render_template('index.html', 
                                 error=error_message,
                                 feature_count=len(feature_names),
                                 model_date=model_date,
                                 feature_stats=feature_stats,
                                 values=form_values,
                                 best_params=best_params)
        
        # Add domain validation
        domain_warnings, out_of_range = validate_input_domain(input_values, feature_stats, label_encoders)
        
        X_input = pd.DataFrame([input_values])
        X_scaled = scaler.transform(X_input)
        
        prediction = model.predict(X_scaled)[0]
        
        # Log warnings for out-of-range predictions
        if domain_warnings:
            logging.warning(f"Prediction with out-of-range inputs: {domain_warnings}")
        
        result = f"{float(prediction):.4f}"
        prediction_time = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        
        logging.info(f"Enhanced prediction: {result}")
        
        return render_template('index.html', 
                             result=result, 
                             prediction_time=prediction_time,
                             feature_count=len(feature_names),
                             model_date=model_date,
                             feature_stats=feature_stats,
                             values=form_values,
                             best_params=best_params,
                             domain_warnings=domain_warnings)
    
    except Exception as e:
        logging.error("Enhanced prediction error: " + str(e))
        form_values = request.form.to_dict()
        return render_template('index.html', 
                             error=f"Prediction Error: {str(e)}",
                             feature_count=len(feature_names),
                             model_date=model_date,
                             feature_stats=feature_stats,
                             values=form_values,
                             best_params=best_params)

@app.route('/static/plots/<path:filename>')
def serve_plots(filename):
    return send_from_directory(Config.PLOTS_DIR, filename)

@app.route('/reload-model', methods=['POST'])
def reload_model():
    """Endpoint to reload the model."""
    try:
        initialize_application(force_retrain=True)
        return jsonify({"status": "success", "message": "Model reloaded successfully"})
    except Exception as e:
        return jsonify({"status": "error", "message": str(e)}), 500

def open_browser():
    webbrowser.open_new("http://127.0.0.1:5000/")

if __name__ == "__main__":
    if model is not None:
        print("\n" + "="*80)
        print("🚀 PROFESSIONAL Flask XGBoost with GridSearchCV - Grout Prediction Server")
        print("📊 Access at: http://127.0.0.1:5000/")
        print(f"🔢 Features: {len(feature_names)} optimized parameters")
        print("⚙️  Optimization: GridSearchCV with hyperparameter tuning")
        print("🎨 Interface: Professional dashboard with advanced analytics")
        print("📈 Visualization: Enhanced charts and model insights")
        print("🔍 Applicability Domain: All features treated as numerical")
        print("⏰ Model date:", model_date)
        print("="*80 + "\n")
        Timer(2.0, open_browser).start()
        app.run(debug=True, use_reloader=False, host='0.0.0.0', port=5000)
    else:
        print("❌ Cannot start server: Professional model initialization failed")
        print("💡 Check the logs for more information about the initialization error.")
        app.run(debug=True, use_reloader=False, host='0.0.0.0', port=5000)

2026-03-03 14:15:43,061 - INFO - Setting up professional Flask application with GridSearchCV...
2026-03-03 14:15:43,064 - INFO - Force retrain mode: Deleting existing model files...
2026-03-03 14:15:43,064 - INFO - Deleted existing file: xgboost_grout_model.pkl
2026-03-03 14:15:43,071 - INFO - Deleted existing file: model_features.pkl
2026-03-03 14:15:43,074 - INFO - Deleted existing file: scaler.pkl
2026-03-03 14:15:43,078 - INFO - Deleted existing file: feature_stats.pkl
2026-03-03 14:15:43,081 - INFO - Deleted existing file: label_encoders.pkl
2026-03-03 14:15:43,083 - INFO - Training enhanced model with GridSearchCV...
2026-03-03 14:15:43,333 - INFO - Loaded dataset with shape: (213, 8)
2026-03-03 14:15:43,373 - INFO - Feature 'depth' treated as numerical with range [5.00, 100.00]
2026-03-03 14:15:43,382 - INFO - Feature 'Q' treated as numerical with range [0.08, 121.33]
2026-03-03 14:15:43,385 - INFO - Feature 'LU' treated as numerical with range [0.50, 100.00]
2026-03-03 14:15:43

Fitting 5 folds for each of 243 candidates, totalling 1215 fits


2026-03-03 14:16:23,702 - INFO - GridSearchCV completed. Best parameters: {'colsample_bytree': 0.8, 'learning_rate': 0.2, 'max_depth': 3, 'n_estimators': 100, 'subsample': 0.9}
2026-03-03 14:16:23,704 - INFO - Best CV score: 138.3301
2026-03-03 14:16:23,713 - INFO - Enhanced model trained and saved with 7 features.
2026-03-03 14:16:23,719 - INFO - Professional HTML template with enhanced UI created!
2026-03-03 14:16:23,719 - INFO - Professional Flask app with enhanced UI ready to launch!



🚀 PROFESSIONAL Flask XGBoost with GridSearchCV - Grout Prediction Server
📊 Access at: http://127.0.0.1:5000/
🔢 Features: 7 optimized parameters
⚙️  Optimization: GridSearchCV with hyperparameter tuning
🎨 Interface: Professional dashboard with advanced analytics
📈 Visualization: Enhanced charts and model insights
🔍 Applicability Domain: All features treated as numerical
⏰ Model date: 2026-03-03 14:16

 * Serving Flask app '__main__'
 * Debug mode: on


2026-03-03 14:16:23,799 - INFO - WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://192.168.0.102:5000
2026-03-03 14:16:23,800 - INFO - Press CTRL+C to quit
2026-03-03 14:16:30,126 - INFO - 127.0.0.1 - - [03/Mar/2026 14:16:30] "GET / HTTP/1.1" 200 -
2026-03-03 14:16:33,337 - INFO - 127.0.0.1 - - [03/Mar/2026 14:16:33] "GET /static/plots/parameter_performance.png HTTP/1.1" 200 -
2026-03-03 14:16:34,344 - INFO - 127.0.0.1 - - [03/Mar/2026 14:16:34] "GET /static/plots/model_evaluation.png HTTP/1.1" 200 -
2026-03-03 14:16:34,421 - INFO - 127.0.0.1 - - [03/Mar/2026 14:16:34] "GET /favicon.ico HTTP/1.1" 404 -
2026-03-03 14:21:02,766 - INFO - Enhanced prediction: 24.8811
2026-03-03 14:21:02,776 - INFO - 127.0.0.1 - - [03/Mar/2026 14:21:02] "POST /predict HTTP/1.1" 200 -
2026-03-03 14:21:02,858 - INFO - 127.0.0.1 - - [03/Mar/2026 14: